# WS1 — OpenActive Data Acquisition & Exploration
**Owner:** Michael · **Items 1 & 3** · **Branch:** `feat/ws1-data-acquisition`

**Goal of this notebook:** harvest London Sport's Open Sessions feed, clean it, filter it to London, and — the key deliverable — **measure how many live sessions there are per LSOA / MSOA / borough and how many small areas have zero**, so the team can decide the unit of analysis for the equity analysis.

**How to use it:** run top to bottom. Two cells need you to act:
- **Section 2** asks you to *look at one real record* and confirm the field paths used in Section 3 (the OpenActive schema can vary by publisher).
- **Section 7** needs the **ONS boundary files from Item 3** (in `data/external/`). Until then, Section 6 gives a rough London estimate so you're not blocked.

**Reminders (assessed practice):** data lives in the git-ignored `data/` folder and is **never committed**; **Clear All Outputs before every commit** (outputs save into the .ipynb and can leak data); commit only the notebook + aggregate CSVs in `reports/`.

## Section 0 — Configuration & setup

In [ ]:
import json, time, random
from pathlib import Path
from datetime import date
from urllib.parse import urljoin

import requests
from requests.adapters import HTTPAdapter
try:
    from urllib3.util import Retry
except Exception:
    from urllib3.util.retry import Retry
import pandas as pd
import numpy as np

RANDOM_SEED = 42          # project convention: fix seeds everywhere
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Find the repo root (the folder containing .git) so paths work wherever the notebook runs.
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / '.git').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

DATA      = REPO_ROOT / 'data'
RAW       = DATA / 'raw'
PROCESSED = DATA / 'processed'
EXTERNAL  = DATA / 'external'
REPORTS   = REPO_ROOT / 'reports'
for p in (RAW, PROCESSED, EXTERNAL, REPORTS):
    p.mkdir(parents=True, exist_ok=True)

# OpenActive / Open Sessions feeds (London Sport's own publishing tool)
SESSION_SERIES_URL = 'https://opensessions.io/api/rpde/session-series'  # recurring activities (what/where/who/price)
EVENTS_URL         = 'https://opensessions.io/api/rpde/events'          # dated instances (when) - not needed for Item 1
HARVEST_DATE       = date.today().isoformat()
LICENCE            = 'CC-BY 4.0 (https://creativecommons.org/licenses/by/4.0/)'

print('Repo root  :', REPO_ROOT)
print('Harvest date:', HARVEST_DATE)
print('Data folder :', DATA, '(git-ignored - never commit)')

In [ ]:
# A polite, resilient HTTP session (retries on transient errors).
def make_session():
    s = requests.Session()
    s.headers.update({'User-Agent': 'ls-openactive-ws1/0.1 (UoB MSc Data Science group project)'})
    retry = Retry(total=5, backoff_factor=1.0, status_forcelist=[429, 500, 502, 503, 504])
    s.mount('https://', HTTPAdapter(max_retries=retry))
    return s

session = make_session()
# quick connectivity check (also your first look at the feed)
r = session.get(SESSION_SERIES_URL, timeout=30)
print('feed status:', r.status_code)
_page = r.json()
print('top-level keys:', list(_page.keys()))
print('items on first page:', len(_page.get('items', [])))
print('licence reported   :', _page.get('license'))

## Section 1 — Harvest the RPDE feed

The feed is **paged**. Each page is `{ "items": [...], "next": "<url>", "license": ... }`. Each item is `{ "id", "modified", "state", "kind", "data" }` where `state` is `"updated"` (real record, with nested `data`) or `"deleted"` (a tombstone — id only). You follow `next` from the oldest record forward; an **empty page means you've caught up to the live edge**. Rebuilding the current state = apply every `updated` (insert/overwrite by id) and every `deleted` (remove by id); whatever remains is live.

In [ ]:
def harvest_feed(start_url, session, max_pages=20000, sleep=0.25, log_every=25):
    # Page through an OpenActive RPDE feed, following 'next' to the live edge.
    # Returns (current_state, stats): current_state maps id -> the full RPDE item.
    current = {}
    seen_updated = seen_deleted = pages = 0
    url, last_url = start_url, None
    while url and url != last_url and pages < max_pages:
        resp = session.get(url, timeout=60)
        resp.raise_for_status()
        payload = resp.json()
        items = payload.get('items', [])
        if not items:
            break                      # empty page = caught up to live
        for item in items:
            item_id = item.get('id')
            if item_id is None:
                continue
            state = item.get('state')
            if state == 'updated':
                current[item_id] = item
                seen_updated += 1
            elif state == 'deleted':
                current.pop(item_id, None)
                seen_deleted += 1
        nxt = payload.get('next')
        if nxt:
            nxt = urljoin(url, nxt)    # handle relative next URLs
        last_url, url = url, nxt
        pages += 1
        if log_every and pages % log_every == 0:
            print(f'  page {pages:>5} | live: {len(current):>6} | updated seen: {seen_updated:>6} | deleted seen: {seen_deleted:>6}')
        if sleep:
            time.sleep(sleep)
    stats = {'pages': pages, 'seen_updated': seen_updated, 'seen_deleted': seen_deleted, 'live_count': len(current)}
    return current, stats

In [ ]:
print('Harvesting SessionSeries feed - this can take a while (many pages)...')
t0 = time.time()
live_items, stats = harvest_feed(SESSION_SERIES_URL, session)
print(f'\nDone in {time.time()-t0:.0f}s')
print(stats)

# Save the raw harvest to the git-ignored data/ folder. NEVER commit this file.
raw_path = RAW / f'session_series_raw_{HARVEST_DATE}.json'
with open(raw_path, 'w') as f:
    json.dump(list(live_items.values()), f)
print('Saved raw harvest ->', raw_path)
print('LIVE SessionSeries records:', len(live_items))

## Section 2 — Inspect the data model  ⚠️ verify field paths

Print one real record and check that the fields Section 3 reads (`name`, `location.geo.latitude/longitude`, `location.name`, `activity[].prefLabel`, `offers[].price`) actually exist where expected. If a publisher nests something differently, adjust the extractors in Section 3.

In [ ]:
records = list(live_items.values())
print('Total live records:', len(records))
if records:
    sample = records[0]
    print('\nRPDE item keys:', list(sample.keys()))
    print('\n--- sample item["data"] (the actual SessionSeries) ---')
    print(json.dumps(sample.get('data', {}), indent=2)[:4000])
else:
    print('No records harvested - re-check Section 1 and the feed status above.')

## Section 3 — Normalise to a flat table

Pull the columns the analysis needs into one tidy DataFrame using safe getters (missing fields become `None`, never an error).

In [ ]:
def g(obj, *keys, default=None):
    # Safely walk nested dicts (and lists by int index). Returns default if any step is missing.
    cur = obj
    for k in keys:
        if isinstance(cur, dict):
            cur = cur.get(k)
        elif isinstance(cur, list) and isinstance(k, int) and -len(cur) <= k < len(cur):
            cur = cur[k]
        else:
            return default
        if cur is None:
            return default
    return cur

def first_activity(data):
    acts = g(data, 'activity', default=[])
    if isinstance(acts, list) and acts:
        return g(acts[0], 'prefLabel') or g(acts[0], 'name')
    if isinstance(acts, dict):
        return g(acts, 'prefLabel') or g(acts, 'name')
    return None

def first_offer_price(data):
    offers = g(data, 'offers', default=[])
    if isinstance(offers, list) and offers:
        return g(offers[0], 'price'), g(offers[0], 'priceCurrency')
    if isinstance(offers, dict):
        return g(offers, 'price'), g(offers, 'priceCurrency')
    return None, None

def extract(item):
    data = item.get('data', {}) or {}
    price, currency = first_offer_price(data)
    acc = g(data, 'accessibilitySupport') or g(data, 'accessibilityInformation')
    return {
        'id'           : item.get('id'),
        'oa_id'        : g(data, '@id'),
        'name'         : g(data, 'name'),
        'activity_type': first_activity(data),
        'organizer'    : g(data, 'organizer', 'name') or g(data, 'organiser', 'name') or g(data, 'provider', 'name'),
        'location_name': g(data, 'location', 'name'),
        'latitude'     : g(data, 'location', 'geo', 'latitude'),
        'longitude'    : g(data, 'location', 'geo', 'longitude'),
        'postcode'     : g(data, 'location', 'address', 'postalCode'),
        'region'       : g(data, 'location', 'address', 'addressRegion'),
        'price'        : price,
        'currency'     : currency,
        'max_capacity' : g(data, 'maximumAttendeeCapacity'),
        'has_access_info': acc is not None,
        'modified'     : item.get('modified'),
    }

df = pd.DataFrame([extract(it) for it in records])
print('Shape:', df.shape)
df.head(10)

## Section 4 — Clean: types, duplicates, coordinates

In [ ]:
before = len(df)

# numeric coercion
for col in ['latitude', 'longitude', 'price', 'max_capacity']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# de-duplicate on the feed id
df = df.drop_duplicates(subset='id').reset_index(drop=True)

# coordinate validity: present, not (0,0), inside a generous Great Britain bounding box
not_null = df['latitude'].notna() & df['longitude'].notna()
not_zero = ~((df['latitude'].abs() < 1e-6) & (df['longitude'].abs() < 1e-6))
in_gb    = df['latitude'].between(49.0, 61.0) & df['longitude'].between(-8.5, 2.0)
df['coord_valid'] = not_null & not_zero & in_gb

# free / low-cost flag
df['is_free'] = df['price'].apply(lambda x: bool(x == 0) if pd.notna(x) else False)

print(f'Rows: {before} -> {len(df)} after de-dup')
print('Valid coordinates :', int(df['coord_valid'].sum()), 'of', len(df),
      f"({df['coord_valid'].mean()*100:.1f}%)")
print('Missing coordinates:', int(df[['latitude','longitude']].isna().any(axis=1).sum()))
df.head()

## Section 5 — Data-quality profile (field completeness)

This is part of your Item 1 / data-quality deliverable: how complete is each field?

In [ ]:
import matplotlib.pyplot as plt

key_fields = ['name','activity_type','organizer','location_name','latitude',
              'longitude','postcode','price','max_capacity','has_access_info']

def pct_present(col):
    return df[col].mean()*100 if col == 'has_access_info' else df[col].notna().mean()*100

completeness = (pd.DataFrame({'field': key_fields,
                              'pct_present': [pct_present(f) for f in key_fields]})
                .sort_values('pct_present', ascending=False))
print(completeness.to_string(index=False))
completeness.to_csv(REPORTS / f'field_completeness_{HARVEST_DATE}.csv', index=False)  # aggregate only - safe to commit

ax = completeness.set_index('field')['pct_present'].plot(kind='barh', figsize=(7,4))
ax.set_xlabel('% present'); ax.set_title('OpenActive SessionSeries - field completeness'); ax.invert_yaxis()
plt.tight_layout(); plt.show()

print('\nTop activity types:'); print(df['activity_type'].value_counts().head(15))
print('\nPrice summary (where present):'); print(df['price'].describe())
print('Free share:', f"{df['is_free'].mean()*100:.1f}%")

## Section 6 — Interim London estimate (bounding box)  ⏱️

A quick ballpark **only**, using a London rectangle — not the real boundary. Use it to share an early live-London figure with the team. Section 7 (ONS boundaries) supersedes it.

In [ ]:
london_bbox = (df['coord_valid']
               & df['latitude'].between(51.28, 51.70)
               & df['longitude'].between(-0.55, 0.30))
print('Approx live London sessions (bounding box):', int(london_bbox.sum()))
print('NOTE: rough rectangle, not the GLA boundary. Replace with Section 7 for the real number.')

## Section 7 — London filter via ONS boundaries  📌 needs Item 3 files

Download 2021 **LAD (boroughs)**, **MSOA**, and **LSOA** boundaries for London from the ONS Open Geography Portal (Item 3), put them in `data/external/`, and set the paths below. Then this filters sessions to London and prepares the geometry for the density analysis.

In [ ]:
import geopandas as gpd

# ---- SET THESE to your actual downloaded filenames (in data/external/) ----
BOROUGH_FILE = EXTERNAL / 'london_lad_2021.geojson'
MSOA_FILE    = EXTERNAL / 'london_msoa_2021.geojson'
LSOA_FILE    = EXTERNAL / 'london_lsoa_2021.geojson'

WORKING_CRS = 'EPSG:27700'   # British National Grid - good for spatial ops in GB

def load_boundary(path):
    if not Path(path).exists():
        print(f'!! Missing: {path}  -> download in Item 3, drop in data/external/, set the path above.')
        return None
    return gpd.read_file(path).to_crs(WORKING_CRS)

boroughs = load_boundary(BOROUGH_FILE)
msoas    = load_boundary(MSOA_FILE)
lsoas    = load_boundary(LSOA_FILE)

# tip: after loading, check the area-code column names, e.g.  boroughs.columns
# ONS 2021 columns are usually LAD22CD/LAD22NM, MSOA21CD, LSOA21CD (year suffix may differ).

# Build a GeoDataFrame of valid-coordinate sessions and project to the working CRS.
sv = df[df['coord_valid']].copy()
sessions_gdf = gpd.GeoDataFrame(
    sv, geometry=gpd.points_from_xy(sv['longitude'], sv['latitude']), crs='EPSG:4326'
).to_crs(WORKING_CRS)
print('Sessions with valid coords:', len(sessions_gdf))

# Filter to London using the borough layer (point within any London LAD).
if boroughs is not None:
    london_sessions = gpd.sjoin(sessions_gdf, boroughs, how='inner', predicate='within')
    keep = [c for c in london_sessions.columns if c == 'geometry' or c in df.columns]
    london_sessions = london_sessions[keep].copy()      # drop join artefacts for clean re-joins
    print('LIVE LONDON SESSIONS (boundary filter):', len(london_sessions))
else:
    london_sessions = None
    print('Add the borough boundary file to compute the real London count.')

## Section 8 — Spatial density & the granularity decision  ⭐ key deliverable

For each level, count sessions per area and check how many areas are **empty** (zero-inflation). High emptiness at LSOA level is the signal to step up to MSOA or borough.

In [ ]:
def density_report(sessions_gdf, area_gdf, code_col, level_name):
    if area_gdf is None or sessions_gdf is None:
        print(f'[{level_name}] skipped - missing inputs.'); return None
    if code_col not in area_gdf.columns:
        print(f'[{level_name}] code column {code_col!r} not found. Columns:', list(area_gdf.columns)); return None
    joined = gpd.sjoin(sessions_gdf, area_gdf[[code_col, 'geometry']], how='inner', predicate='within')
    per_area = joined.groupby(code_col).size()
    full = per_area.reindex(area_gdf[code_col].unique(), fill_value=0)   # include empty areas
    total = int(area_gdf[code_col].nunique())
    with_s = int((full > 0).sum())
    row = {'level': level_name, 'total_areas': total, 'areas_with_sessions': with_s,
           'areas_with_zero': total - with_s, 'pct_zero': round((total-with_s)/total*100, 1),
           'median_per_area': float(full.median()), 'mean_per_area': round(float(full.mean()), 2),
           'max_per_area': int(full.max())}
    return row, full

levels = [(boroughs, 'LAD22CD', 'Borough (LAD)'),   # <-- adjust code columns to your files
          (msoas,    'MSOA21CD', 'MSOA'),
          (lsoas,    'LSOA21CD', 'LSOA')]

results, series_by_level = [], {}
for area_gdf, code_col, name in levels:
    out = density_report(london_sessions, area_gdf, code_col, name)
    if out:
        row, full = out
        results.append(row); series_by_level[name] = full

if results:
    summary = pd.DataFrame(results)
    print(summary.to_string(index=False))
    summary.to_csv(REPORTS / f'granularity_summary_{HARVEST_DATE}.csv', index=False)
else:
    print('No density results yet - add boundary files and fix the code-column names above.')

In [ ]:
# Visuals: a borough choropleth (easy to read) and an LSOA histogram (shows the zero spike).
if boroughs is not None and london_sessions is not None and 'LAD22CD' in boroughs.columns:
    bc = gpd.sjoin(london_sessions, boroughs[['LAD22CD','geometry']], predicate='within').groupby('LAD22CD').size()
    bmap = boroughs.merge(bc.rename('sessions'), left_on='LAD22CD', right_index=True, how='left')
    bmap['sessions'] = bmap['sessions'].fillna(0)
    ax = bmap.plot(column='sessions', legend=True, figsize=(8,7), cmap='viridis', edgecolor='white', linewidth=0.3)
    ax.set_title(f'Live OpenActive sessions per borough ({HARVEST_DATE})'); ax.axis('off')
    plt.tight_layout(); plt.show()

if 'LSOA' in series_by_level:
    series_by_level['LSOA'].plot(kind='hist', bins=40, figsize=(7,4))
    plt.xlabel('sessions per LSOA'); plt.title('Sessions per LSOA (note the spike at 0 = zero-inflation)')
    plt.tight_layout(); plt.show()

## Section 9 — Recommendation

A **guideline** (not a hard rule) plus a draft sentence for the data-quality write-up. The team makes the final call with the supervisor.

In [ ]:
def suggest_unit(summary_df):
    if summary_df is None or summary_df.empty:
        return 'Run Section 8 first.'
    lines = []
    for _, r in summary_df.iterrows():
        ok = (r['pct_zero'] < 60) and (r['median_per_area'] >= 1)
        lines.append(f"- {r['level']}: {int(r['areas_with_sessions'])}/{int(r['total_areas'])} populated, "
                     f"{r['pct_zero']}% empty, median {r['median_per_area']:.0f}/area "
                     f"-> {'viable' if ok else 'fragile (sparse / zero-inflated)'}")
    return '\n'.join(lines)

if results:
    print('Granularity guideline (finalise with the team / supervisor):\n')
    print(suggest_unit(pd.DataFrame(results)))
    print('\nDraft sentence for the data-quality write-up (fill in the numbers):')
    print('"From a harvest of <N> live London SessionSeries on', HARVEST_DATE,
          ', spreading sessions across <X> LSOAs leaves <Y>% with zero sessions (median <Z>/area).')
    print(' We therefore adopt the <LSOA / MSOA / borough> level as the unit of analysis for projection and')
    print(' clustering, and report sparsity as a substantive finding."')

## Section 10 — Save outputs & commit checklist

In [ ]:
# Save the processed London dataset to the git-ignored data/ folder (do NOT commit).
if london_sessions is not None:
    london_df = pd.DataFrame(london_sessions.drop(columns='geometry'))
    out = PROCESSED / f'london_sessions_{HARVEST_DATE}.parquet'
    try:
        london_df.to_parquet(out, index=False)
    except Exception:
        out = PROCESSED / f'london_sessions_{HARVEST_DATE}.csv'
        london_df.to_csv(out, index=False)
    print('Saved processed London sessions ->', out, f'({len(london_df)} rows)')

print('\nCOMMIT CHECKLIST')
print('1. Notebook menu -> Clear All Outputs   (outputs save into the .ipynb and can leak data)')
print('2. git status     -> confirm NOTHING under data/ is staged')
print('3. Commit only    -> this notebook + reports/*.csv (aggregate numbers) + any notes')
print('4. git push -u origin feat/ws1-data-acquisition  -> open a PR -> request review (no self-merge)')